In [ ]:
pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviews") \
    .getOrCreate()

df = spark.read.json("/content/Grocery_and_Gourmet_Food.jsonl.gz")

df.show(5)
df.printSchema()

+----------+------------+------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|      asin|helpful_vote|images|parent_asin|rating|                text|    timestamp|               title|             user_id|verified_purchase|
+----------+------------+------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|B00CM36GAQ|           0|    []| B00CM36GAQ|   5.0|Excellent!! Yummy...|1587854482395|  Excellent!  Yummy!|AFKZENTNBQ7A7V7UX...|             true|
|B074J5WVYH|           0|    []| B0759B7KLH|   5.0|Excellent!  The b...|1587854400380|   Delicious!!! Yum!|AFKZENTNBQ7A7V7UX...|             true|
|B079TRNVHX|           1|    []| B079TRNVHX|   5.0|These are very ta...|1587853224527|Extremely Delicio...|AFKZENTNBQ7A7V7UX...|             true|
|B07194LN2Z|           0|    []| B07194LN2Z|   5.0|        My favorite!|1581313319614|          Delicious!|AFKZENTNBQ7

In [ ]:
from pyspark.sql.functions import col

df_clean = df.dropna(subset=["asin", "rating", "user_id", "text"])

df_clean = df_clean.select(
    "asin",
    "user_id",
    "rating",
    "helpful_vote",
    "verified_purchase",
    "text",
    "timestamp"
)

df_clean.show(5)

+----------+--------------------+------+------------+-----------------+--------------------+-------------+
|      asin|             user_id|rating|helpful_vote|verified_purchase|                text|    timestamp|
+----------+--------------------+------+------------+-----------------+--------------------+-------------+
|B00CM36GAQ|AFKZENTNBQ7A7V7UX...|   5.0|           0|             true|Excellent!! Yummy...|1587854482395|
|B074J5WVYH|AFKZENTNBQ7A7V7UX...|   5.0|           0|             true|Excellent!  The b...|1587854400380|
|B079TRNVHX|AFKZENTNBQ7A7V7UX...|   5.0|           1|             true|These are very ta...|1587853224527|
|B07194LN2Z|AFKZENTNBQ7A7V7UX...|   5.0|           0|             true|        My favorite!|1581313319614|
|B005CD4196|AFKZENTNBQ7A7V7UX...|   5.0|           7|             true|Great for making ...|1581313294965|
+----------+--------------------+------+------------+-----------------+--------------------+-------------+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import from_unixtime

df_clean = df_clean.withColumn(
    "review_date",
    from_unixtime(col("timestamp")/1000).cast("timestamp")
)

df_clean.select("timestamp","review_date").show(5)

+-------------+-------------------+
|    timestamp|        review_date|
+-------------+-------------------+
|1587854482395|2020-04-25 22:41:22|
|1587854400380|2020-04-25 22:40:00|
|1587853224527|2020-04-25 22:20:24|
|1581313319614|2020-02-10 05:41:59|
|1581313294965|2020-02-10 05:41:34|
+-------------+-------------------+
only showing top 5 rows


In [ ]:
rating_dist = df_clean.groupBy("rating").count().orderBy("rating")

rating_dist.show()

+------+-------+
|rating|  count|
+------+-------+
|   0.0|      1|
|   1.0|1855049|
|   2.0| 695428|
|   3.0| 898317|
|   4.0|1256795|
|   5.0|9612930|
+------+-------+



In [ ]:
from pyspark.sql.functions import count

top_products = df_clean.groupBy("asin") \
    .agg(count("*").alias("review_count")) \
    .orderBy(col("review_count").desc())

top_products.show(10)

+----------+------------+
|      asin|review_count|
+----------+------------+
|B00DS842HS|       15582|
|B006CQ1ZHI|       15507|
|B08SJ4HVSY|        7962|
|B01N22CM3F|        7912|
|B0131A6FJA|        7375|
|B00PFDH0IC|        7211|
|B079V8CKDM|        7202|
|B007Y59HVM|        6967|
|B00FJYTGBG|        6860|
|B007TGDXNO|        6723|
+----------+------------+
only showing top 10 rows


In [ ]:
from pyspark.sql.functions import avg

product_rating = df_clean.groupBy("asin") \
    .agg(avg("rating").alias("avg_rating"),
         count("*").alias("total_reviews")) \
    .orderBy(col("total_reviews").desc())

product_rating.show(10)

+----------+------------------+-------------+
|      asin|        avg_rating|total_reviews|
+----------+------------------+-------------+
|B00DS842HS| 4.674367860351688|        15582|
|B006CQ1ZHI| 4.532920616495776|        15507|
|B08SJ4HVSY| 4.663275558904798|         7962|
|B01N22CM3F|4.3655207280080885|         7912|
|B0131A6FJA|3.9213559322033897|         7375|
|B00PFDH0IC| 4.293024545832756|         7211|
|B079V8CKDM|  4.33462926964732|         7202|
|B007Y59HVM| 4.556767618774221|         6967|
|B00FJYTGBG| 4.580466472303207|         6860|
|B007TGDXNO| 4.300163617432694|         6723|
+----------+------------------+-------------+
only showing top 10 rows


In [ ]:
verified_stats = df_clean.groupBy("verified_purchase") \
    .count()

verified_stats.show()

+-----------------+--------+
|verified_purchase|   count|
+-----------------+--------+
|             true|13176519|
|            false| 1142001|
+-----------------+--------+



In [ ]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF

tokenizer = Tokenizer(inputCol="text", outputCol="words")

wordsData = tokenizer.transform(df)

remover = StopWordsRemover(inputCol="words", outputCol="filtered")

filteredData = remover.transform(wordsData)

hashingTF = HashingTF(
    inputCol="filtered",
    outputCol="text_features",
    numFeatures=1000
)

featuredData = hashingTF.transform(filteredData)

In [ ]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF

# Tokenize review text
tokenizer = Tokenizer(inputCol="text", outputCol="words")
wordsData = tokenizer.transform(df)

# Remove stop words
remover = StopWordsRemover(inputCol="words", outputCol="filtered")
filteredData = remover.transform(wordsData)

# Convert words to numerical features
hashingTF = HashingTF(
    inputCol="filtered",
    outputCol="raw_features",
    numFeatures=1000
)

tfData = hashingTF.transform(filteredData)

# TF-IDF weighting
idf = IDF(inputCol="raw_features", outputCol="text_features")

idfModel = idf.fit(tfData)
featuredData = idfModel.transform(tfData)

In [ ]:
from pyspark.sql.functions import length

featuredData = featuredData.withColumn(
    "review_length",
    length("text")
)

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "verified_purchase",
        "helpful_vote",
        "review_length",
        "text_features"
    ],
    outputCol="features"
)

final_data = assembler.transform(featuredData)

In [ ]:
train, test = final_data.randomSplit([0.8, 0.2], seed=42)

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviewSample") \
    .getOrCreate()

data = [
("B00CM36GAQ",0,"B00CM36GAQ",5.0,"Excellent!! Yummy snack",1587854482395,"Excellent Yummy","U1",True),
("B074J5WVYH",0,"B0759B7KLH",5.0,"Excellent taste and quality",1587854400380,"Delicious Yum","U2",True),
("B079TRNVHX",1,"B079TRNVHX",5.0,"Very tasty and fresh product",1587853224527,"Extremely Delicious","U3",True),
("B07194LN2Z",0,"B07194LN2Z",4.0,"My favorite snack",1581313319614,"Delicious","U4",True),
("B005CD4196",7,"B005CD4196",5.0,"Great for making desserts",1581313294965,"Great taste","U5",True),
("B00TEST111",2,"B00TEST111",3.0,"Average product quality",1581313200000,"Okay product","U6",False),
("B00TEST222",1,"B00TEST222",2.0,"Not very good taste",1581313100000,"Bad taste","U7",False),
("B00TEST333",0,"B00TEST333",1.0,"Terrible product never buy",1581313000000,"Worst product","U8",False)
]

columns = [
"asin",
"helpful_vote",
"parent_asin",
"rating",
"text",
"timestamp",
"title",
"user_id",
"verified_purchase"
]

df = spark.createDataFrame(data, columns)

df.show()
df.printSchema()

+----------+------------+-----------+------+--------------------+-------------+-------------------+-------+-----------------+
|      asin|helpful_vote|parent_asin|rating|                text|    timestamp|              title|user_id|verified_purchase|
+----------+------------+-----------+------+--------------------+-------------+-------------------+-------+-----------------+
|B00CM36GAQ|           0| B00CM36GAQ|   5.0|Excellent!! Yummy...|1587854482395|    Excellent Yummy|     U1|             true|
|B074J5WVYH|           0| B0759B7KLH|   5.0|Excellent taste a...|1587854400380|      Delicious Yum|     U2|             true|
|B079TRNVHX|           1| B079TRNVHX|   5.0|Very tasty and fr...|1587853224527|Extremely Delicious|     U3|             true|
|B07194LN2Z|           0| B07194LN2Z|   4.0|   My favorite snack|1581313319614|          Delicious|     U4|             true|
|B005CD4196|           7| B005CD4196|   5.0|Great for making ...|1581313294965|        Great taste|     U5|           

In [ ]:
from pyspark.sql.functions import length

df = df.withColumn("review_length", length("text"))

In [ ]:
from pyspark.sql.functions import col

df = df.withColumn(
    "verified_numeric",
    col("verified_purchase").cast("integer")
)

In [ ]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF

tokenizer = Tokenizer(inputCol="text", outputCol="words")
wordsData = tokenizer.transform(df)

remover = StopWordsRemover(inputCol="words", outputCol="filtered")
filteredData = remover.transform(wordsData)

hashingTF = HashingTF(
    inputCol="filtered",
    outputCol="raw_features",
    numFeatures=100
)

tfData = hashingTF.transform(filteredData)

idf = IDF(inputCol="raw_features", outputCol="text_features")
idfModel = idf.fit(tfData)

featuredData = idfModel.transform(tfData)

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "verified_numeric",
        "helpful_vote",
        "review_length",
        "text_features"
    ],
    outputCol="features"
)

final_data = assembler.transform(featuredData)

In [ ]:
train, test = final_data.randomSplit([0.8,0.2], seed=42)

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="rating"
)

lr_model = lr.fit(train)

lr_predictions = lr_model.transform(test)

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="rating"
)

dt_model = dt.fit(train)

dt_predictions = dt_model.transform(test)

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="rating",
    numTrees=10
)

rf_model = rf.fit(train)

rf_predictions = rf_model.transform(test)

In [ ]:
from pyspark.sql.functions import when

final_data = final_data.withColumn(
    "label",
    when(final_data.rating >= 4, 1).otherwise(0)
)

In [ ]:
train, test = final_data.randomSplit([0.8, 0.2], seed=42)

In [ ]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=10
)

gbt_model = gbt.fit(train)

gbt_predictions = gbt_model.transform(test)

gbt_predictions.select("label","prediction").show()

+-----+----------+
|label|prediction|
+-----+----------+
|    1|       1.0|
|    1|       1.0|
+-----+----------+



In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

print("GBT AUC:", evaluator.evaluate(gbt_predictions))

GBT AUC: 1.0


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="rating",
    predictionCol="prediction",
    metricName="accuracy"
)

print("LR Accuracy:", evaluator.evaluate(lr_predictions))
print("DT Accuracy:", evaluator.evaluate(dt_predictions))
print("RF Accuracy:", evaluator.evaluate(rf_predictions))
print("GBT Accuracy:", evaluator.evaluate(gbt_predictions))

LR Accuracy: 0.0
DT Accuracy: 1.0
RF Accuracy: 0.0
GBT Accuracy: 0.0


In [ ]:
from pyspark.sql.functions import avg, count

# Dashboard 1 – Rating Distribution
dashboard1 = df.groupBy("rating").count()

# Dashboard 2 – Average Rating per Product
dashboard2 = df.groupBy("asin").agg(avg("rating").alias("avg_rating"))

# Dashboard 3 – Review Count per Product
dashboard3 = df.groupBy("asin").agg(count("text").alias("total_reviews"))

# Dashboard 4 – User Review Activity
dashboard4 = df.groupBy("user_id").agg(count("text").alias("reviews_written"))

In [ ]:
dashboard1.toPandas().to_csv("dashboard1_dataset.csv", index=False)
dashboard2.toPandas().to_csv("dashboard2_dataset.csv", index=False)
dashboard3.toPandas().to_csv("dashboard3_dataset.csv", index=False)
dashboard4.toPandas().to_csv("dashboard4_dataset.csv", index=False)